In [3]:
import os, json, uuid, httpx
from pathlib import Path
from datetime import datetime
 
import pytesseract
import cv2
import numpy as np
from PIL import Image

In [4]:
try:
    from pdf2image import convert_from_path
    PDF_SUPPORT = True
except ImportError:
    PDF_SUPPORT = False
    print("⚠️  pdf2image not installed — PDF support disabled. Install poppler for Windows.")

In [5]:
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

In [6]:
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2.5:3b"
OUTPUT_DIR   = Path("ocr_output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [7]:
def load_document(file_path: str) -> list:
    path = Path(file_path)
    ext  = path.suffix.lower()
 
    if ext == ".pdf":
        if not PDF_SUPPORT:
            raise RuntimeError("pdf2image not available. Install poppler.")
        # poppler_path needed on Windows — adjust if needed
        # Download from: https://github.com/oschwartz10612/poppler-windows/releases
        # Then set: poppler_path=r"C:\poppler\Library\bin"
        pages = convert_from_path(str(path), dpi=300, poppler_path=r"C:\poppler\poppler-25.12.0\Library\bin")
        print(f"  📄 PDF loaded: {len(pages)} page(s)")
        return pages
 
    elif ext in [".png", ".jpg", ".jpeg", ".tiff", ".bmp"]:
        img = Image.open(path).convert("RGB")
        print(f"  🖼️  Image loaded: {img.size}")
        return [img]
 
    else:
        raise ValueError(f"Unsupported file type: {ext}")
 

In [8]:
def preprocess(pil_img: Image.Image) -> Image.Image:
    """
    Grayscale → Denoise → Adaptive threshold
    Makes text sharp and clean for Tesseract
    """
    # PIL → OpenCV
    cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    gray   = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
 
    # Denoise
    denoised = cv2.fastNlMeansDenoising(gray, h=10)
 
    # Adaptive threshold — handles uneven lighting in scans
    thresh = cv2.adaptiveThreshold(
        denoised, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )
 
    return Image.fromarray(thresh)
 

In [9]:
def run_ocr(pages: list) -> str:
    """Runs Tesseract on all pages, returns combined raw text."""
    config = "--oem 3 --psm 6"
    texts  = []
 
    for i, page in enumerate(pages):
        processed = preprocess(page)
        text      = pytesseract.image_to_string(processed, config=config)
        texts.append(text)
        print(f"  📝 Page {i+1}: {len(text)} chars extracted")
 
    return "\n".join(texts)
 

In [10]:
PROMPTS = {
 
    "PAN_CARD": """
Extract fields from this Company PAN Card OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "pan_number":    "10-char PAN e.g. AAKCM1234C",
  "entity_name":   "Full company name",
  "date_of_reg":   "DD/MM/YYYY",
  "entity_type":   "COMPANY or INDIVIDUAL",
  "issuing_auth":  "Income Tax Department"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "GST_CERTIFICATE": """
Extract fields from this GST Registration Certificate OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "gstin":              "15-char GSTIN",
  "pan_number":         "10-char PAN",
  "legal_name":         "Legal name of business",
  "trade_name":         "Trade name if different",
  "state_code":         "2-digit state code",
  "state":              "State name",
  "status":             "ACTIVE or CANCELLED",
  "registration_date":  "DD/MM/YYYY",
  "constitution":       "Private Limited etc",
  "address":            "Full address",
  "pincode":            "6-digit PIN"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "LEI_CERTIFICATE": """
Extract fields from this LEI Certificate OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "lei_code":           "20-char LEI",
  "legal_name":         "Legal name",
  "cin":                "CIN number",
  "pan_number":         "PAN number",
  "status":             "ACTIVE or INACTIVE",
  "registration_date":  "DD/MM/YYYY",
  "renewal_date":       "DD/MM/YYYY",
  "issuing_lou":        "Issuing organization",
  "country":            "Country"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "INCORPORATION_CERTIFICATE": """
Extract fields from this Certificate of Incorporation OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "company_name":       "Full company name",
  "cin":                "CIN number",
  "pan_number":         "PAN number",
  "date_of_incorp":     "DD/MM/YYYY",
  "company_type":       "Private Limited etc",
  "authorized_capital": "Amount in numbers only e.g. 2500000",
  "state":              "State name",
  "roc":                "ROC office",
  "address":            "Registered address",
  "pincode":            "6-digit PIN"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "MOA": """
Extract fields from this Memorandum of Association OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "company_name":       "Full company name",
  "cin":                "CIN number",
  "state":              "State of registered office",
  "address":            "Registered address",
  "pincode":            "6-digit PIN",
  "authorized_capital": "Amount in numbers only",
  "main_objects":       ["list of main business objects"],
  "subscribers": [
    {"name": "subscriber name", "shares": "number of shares"}
  ]
}
 
OCR TEXT:
{ocr_text}
""",
 
    "AOA": """
Extract fields from this Articles of Association OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "company_name":       "Full company name",
  "cin":                "CIN number",
  "authorized_capital": "Amount in numbers only",
  "min_directors":      "minimum number",
  "max_directors":      "maximum number",
  "directors": [
    {"name": "director name", "din": "DIN number"}
  ]
}
 
OCR TEXT:
{ocr_text}
""",
 
    "REGISTERED_ADDRESS": """
Extract fields from this Registered Address / INC-22 OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "company_name":    "Full company name",
  "cin":             "CIN number",
  "pan_number":      "PAN number",
  "gstin":           "GSTIN",
  "lei":             "LEI code",
  "address_line1":   "Building/flat",
  "address_line2":   "Street/road",
  "area":            "Area/locality",
  "city":            "City",
  "state":           "State",
  "pincode":         "6-digit PIN",
  "srn":             "Service request number",
  "filing_date":     "DD/MM/YYYY",
  "approval_date":   "DD/MM/YYYY",
  "status":          "APPROVED or PENDING"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "ELECTRICITY_BILL": """
Extract fields from this Electricity Bill OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "consumer_name":    "Name on bill",
  "consumer_number":  "Consumer/account number",
  "discom":           "Electricity provider name",
  "address":          "Consumer address",
  "pincode":          "6-digit PIN",
  "bill_number":      "Bill reference number",
  "bill_date":        "DD/MM/YYYY",
  "due_date":         "DD/MM/YYYY",
  "billing_period":   "Month Year",
  "units_consumed":   "Number only",
  "total_amount":     "Number only e.g. 4693.70",
  "connection_type":  "Commercial or Residential"
}
 
OCR TEXT:
{ocr_text}
""",
 
    "TELEPHONE_BILL": """
Extract fields from this Telephone/Landline Bill OCR text.
Return ONLY valid JSON, no explanation, no markdown.
 
Required fields:
{
  "account_name":     "Name on bill",
  "account_number":   "Account number",
  "telephone_number": "Phone number",
  "provider":         "Telecom provider name",
  "address":          "Address on bill",
  "pincode":          "6-digit PIN",
  "bill_number":      "Bill reference number",
  "bill_date":        "DD/MM/YYYY",
  "due_date":         "DD/MM/YYYY",
  "billing_period":   "Month Year",
  "total_amount":     "Number only",
  "connection_type":  "Landline or Mobile"
}
 
OCR TEXT:
{ocr_text}
"""
}

In [11]:
def extract_with_llm(ocr_text: str, doc_type: str) -> dict:
    """Sends OCR text to Qwen3 via Ollama, returns parsed JSON."""
 
    prompt_template = PROMPTS.get(doc_type)
    if not prompt_template:
        raise ValueError(f"No prompt defined for doc_type: {doc_type}")
 
    prompt = prompt_template.replace("{ocr_text}", ocr_text)
 
    print(f"  🤖 Sending to Qwen3...")
 
    try:
        response = httpx.post(
            OLLAMA_URL,
            json={
                "model":  OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                # Tell Qwen3 to think less, just extract — faster
                "options": {
                    "temperature": 0.1,
                    "top_p": 0.9,
                }
            },
            timeout=120.0  # Qwen3 can be slow on CPU
        )
        response.raise_for_status()
        raw_response = response.json()["response"]
 
        # ── Clean up response ─────────────────────────────────
        # Qwen3 sometimes wraps in ```json ... ```
        cleaned = raw_response.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("```")[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:]
        cleaned = cleaned.strip()
 
        # Also strip <think>...</think> blocks if present
        if "<think>" in cleaned:
            import re
            cleaned = re.sub(r"<think>.*?</think>", "", cleaned, flags=re.DOTALL).strip()
 
        parsed = json.loads(cleaned)
        print(f"  ✅ JSON extracted: {len(parsed)} fields")
        return parsed
 
    except httpx.ConnectError:
        print("  ❌ Ollama not running. Start with: ollama serve")
        return {"error": "OLLAMA_NOT_RUNNING"}
    except json.JSONDecodeError as e:
        print(f"  ⚠️  JSON parse failed: {e}")
        print(f"  Raw response: {raw_response[:200]}")
        return {"error": "JSON_PARSE_FAILED", "raw": raw_response[:500]}
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return {"error": str(e)}
 

In [12]:
def build_document_json(
    file_path: str,
    doc_type:  str,
    fields:    dict,
    ocr_text:  str
) -> dict:
    """Wraps extracted fields in the standard compliance JSON envelope."""
 
    # Cross-check fields — same keys across ALL documents
    # Used for consistency verification later
    cross_check = {
        "pan_number":   fields.get("pan_number")   or fields.get("pan"),
        "company_name": fields.get("entity_name")  or fields.get("legal_name")   or
                        fields.get("company_name") or fields.get("consumer_name") or
                        fields.get("account_name"),
        "address":      fields.get("address")      or
                        f"{fields.get('address_line1','')} {fields.get('address_line2','')}".strip(),
        "pincode":      fields.get("pincode"),
        "cin":          fields.get("cin"),
        "gstin":        fields.get("gstin"),
    }
 
    # Remove None values
    cross_check = {k: v for k, v in cross_check.items() if v}
 
    return {
        "doc_id":       str(uuid.uuid4())[:8].upper(),
        "doc_type":     doc_type,
        "source_file":  Path(file_path).name,
        "extracted_at": datetime.now().isoformat(),
        "fields":       fields,
        "cross_check":  cross_check,
        "ocr_raw_text": ocr_text[:1000],  # first 1000 chars for debug
        "status":       "FAILED" if "error" in fields else "EXTRACTED"
    }

In [13]:
def process_document(file_path: str, doc_type: str) -> dict:
    """Full pipeline for a single document."""
 
    print(f"\n{'─'*55}")
    print(f"  Processing: {Path(file_path).name}")
    print(f"  Doc Type  : {doc_type}")
    print(f"{'─'*55}")
 
    # Step 1 — Load
    pages = load_document(file_path)
 
    # Step 2+3 — Preprocess + OCR
    ocr_text = run_ocr(pages)
 
    # Step 4 — LLM extraction
    fields = extract_with_llm(ocr_text, doc_type)
 
    # Step 5 — Build final JSON
    doc_json = build_document_json(file_path, doc_type, fields, ocr_text)
 
    # Save to output folder
    out_file = OUTPUT_DIR / f"{doc_type}_{doc_json['doc_id']}.json"
    with open(out_file, "w") as f:
        json.dump(doc_json, f, indent=2)
 
    print(f"  💾 Saved: {out_file}")
    return doc_json

In [14]:
DOCUMENTS = [
    ("01_Company_PAN_Card.png",              "PAN_CARD"),
    ("02_GST_Certificate.pdf",               "GST_CERTIFICATE"),
    ("03_LEI_Certificate.pdf",               "LEI_CERTIFICATE"),
    ("04_Certificate_of_Incorporation.pdf",  "INCORPORATION_CERTIFICATE"),
    ("05_MOA.pdf",                           "MOA"),
    ("06_AOA.pdf",                           "AOA"),
    ("07_Registered_Address_INC22.pdf",      "REGISTERED_ADDRESS"),
    ("08_Electricity_Bill.pdf",              "ELECTRICITY_BILL"),
    ("09_Telephone_Landline_Bill.pdf",       "TELEPHONE_BILL"),
]
 


In [15]:
def normalize(value: str) -> str:
    """Normalize string for comparison — lowercase, strip extra spaces."""
    if not value:
        return ""
    return " ".join(str(value).lower().strip().split())

In [16]:
def run_cross_checks(results: list) -> dict:
    """
    Full cross-check engine.
    Checks 6 critical parameters across all documents that should contain them.
    Pinpoints exactly which document has the inconsistent value.
    """
 
    # Parameters to check → their key in cross_check block
    CHECKS = {
        "PAN Number":   "pan_number",
        "Company Name": "company_name",
        "Pincode":      "pincode",
        "CIN":          "cin",
        "GSTIN":        "gstin",
        "Address":      "address",
    }
 
    # Which doc types are expected to have each field
    FIELD_PRESENCE = {
        "pan_number": [
            "PAN_CARD", "GST_CERTIFICATE", "LEI_CERTIFICATE",
            "INCORPORATION_CERTIFICATE", "REGISTERED_ADDRESS"
        ],
        "company_name": [
            "PAN_CARD", "GST_CERTIFICATE", "LEI_CERTIFICATE",
            "INCORPORATION_CERTIFICATE", "MOA", "AOA",
            "REGISTERED_ADDRESS", "ELECTRICITY_BILL", "TELEPHONE_BILL"
        ],
        "pincode": [
            "GST_CERTIFICATE", "INCORPORATION_CERTIFICATE",
            "REGISTERED_ADDRESS", "ELECTRICITY_BILL", "TELEPHONE_BILL"
        ],
        "cin": [
            "LEI_CERTIFICATE", "INCORPORATION_CERTIFICATE",
            "MOA", "AOA", "REGISTERED_ADDRESS"
        ],
        "gstin": [
            "GST_CERTIFICATE", "REGISTERED_ADDRESS"
        ],
        "address": [
            "GST_CERTIFICATE", "INCORPORATION_CERTIFICATE",
            "REGISTERED_ADDRESS", "ELECTRICITY_BILL", "TELEPHONE_BILL"
        ],
    }
 
    report = {"passed": [], "failed": [], "warnings": []}
 
    print("\n" + "=" * 65)
    print("   CROSS-CHECK REPORT")
    print("=" * 65)
 
    for label, key in CHECKS.items():
        expected_docs = FIELD_PRESENCE.get(key, [])
 
        found        = {}  # doc_type -> raw value
        missing_from = []
 
        for r in results:
            if r["doc_type"] not in expected_docs:
                continue
            val = r["cross_check"].get(key)
            if val:
                found[r["doc_type"]] = val
            else:
                missing_from.append(r["doc_type"])
 
        print(f"\n  {'-' * 60}")
        print(f"  Checking : {label}")
 
        if not found:
            msg = "Not extracted from any document — possible OCR failure"
            print(f"  WARNING  : {msg}")
            report["warnings"].append({"parameter": label, "issue": msg})
            continue
 
        normalized  = {doc: normalize(val) for doc, val in found.items()}
        unique_vals = set(normalized.values())
 
        if len(unique_vals) == 1:
            print(f"  RESULT   : CONSISTENT  ->  {list(found.values())[0]}")
            for doc in found:
                print(f"     {doc:<44} OK")
            report["passed"].append({
                "parameter": label,
                "value":     list(found.values())[0],
                "docs":      list(found.keys())
            })
 
        else:
            print(f"  RESULT   : MISMATCH DETECTED")
 
            # Group docs by their normalized value
            value_groups = {}
            for doc, val in found.items():
                norm = normalize(val)
                value_groups.setdefault(norm, []).append((doc, val))
 
            # Majority = most docs agree on this value
            majority_norm = max(value_groups, key=lambda x: len(value_groups[x]))
            majority_val  = value_groups[majority_norm][0][1]
 
            for norm_val, doc_list in value_groups.items():
                is_majority = (norm_val == majority_norm)
                tag = "majority (expected)" if is_majority else "INCONSISTENT"
                for doc, raw_val in doc_list:
                    print(f"     {doc:<44} {tag}")
                    print(f"       Value: \"{raw_val}\"")
 
            inconsistent_docs = [
                doc
                for norm_val, doc_list in value_groups.items()
                if norm_val != majority_norm
                for doc, _ in doc_list
            ]
            inconsistent_vals = [
                val
                for norm_val, doc_list in value_groups.items()
                if norm_val != majority_norm
                for _, val in doc_list
            ]
 
            print(f"\n  ACTION REQUIRED:")
            print(f"    Kindly check the following document(s) : {inconsistent_docs}")
            print(f"    Check consistency for                  : [{label}]")
            print(f"    Inconsistent value(s) found            : {inconsistent_vals}")
            print(f"    Expected value (majority)              : \"{majority_val}\"")
 
            report["failed"].append({
                "parameter":           label,
                "majority_value":      majority_val,
                "inconsistent_docs":   inconsistent_docs,
                "inconsistent_values": inconsistent_vals,
                "all_values":          found
            })
 
        if missing_from:
            print(f"  WARNING  : Field not extracted from (possible OCR issue): {missing_from}")
            report["warnings"].append({
                "parameter":    label,
                "missing_from": missing_from,
                "issue":        "Expected but not extracted — verify manually"
            })
 
    # Final verdict
    print(f"\n  {'=' * 65}")
    print(f"  FINAL VERDICT")
    print(f"  {'=' * 65}")
    print(f"  Passed   : {len(report['passed'])} parameter(s)")
    print(f"  Failed   : {len(report['failed'])} parameter(s)")
    print(f"  Warnings : {len(report['warnings'])} parameter(s)")
 
    if report["failed"]:
        print(f"\n  INCONSISTENCIES FOUND — Review before proceeding:")
        for item in report["failed"]:
            print(f"    [{item['parameter']}]  ->  inconsistent in: {item['inconsistent_docs']}")
        print(f"\n  Status: COMPLIANCE FAILED — Manual review required")
    elif report["warnings"]:
        print(f"\n  Status: PARTIAL — Some fields missing, verify manually")
    else:
        print(f"\n  Status: ALL CHECKS PASSED — Documents are consistent")
 
    print(f"  {'=' * 65}")
    return report
 


In [17]:
def run_all():
    print("\n" + "=" * 65)
    print("   NBFC OCR PIPELINE — Processing all documents")
    print("=" * 65)
 
    results = []
    failed  = []
 
    for filename, doc_type in DOCUMENTS:
        if not Path(filename).exists():
            print(f"\n  File not found: {filename} — skipping")
            failed.append(filename)
            continue
        result = process_document(filename, doc_type)
        results.append(result)
 
    # Processing summary
    print("\n" + "=" * 65)
    print("   PROCESSING SUMMARY")
    print("=" * 65)
    print(f"  Total Documents : {len(DOCUMENTS)}")
    print(f"  Processed       : {len(results)}")
    print(f"  Not Found       : {len(failed)}")
 
    extracted = [r for r in results if r["status"] == "EXTRACTED"]
    errors    = [r for r in results if r["status"] == "FAILED"]
    print(f"  Extracted OK    : {len(extracted)}")
    print(f"  LLM Errors      : {len(errors)}")
 
    if errors:
        print(f"\n  Documents with extraction errors:")
        for e in errors:
            print(f"    {e['doc_type']} ({e['source_file']})")
 
    # Run full cross-check
    cross_check_report = run_cross_checks(results)
 
    # Save both outputs
    master_file = OUTPUT_DIR / "all_documents.json"
    with open(master_file, "w") as f:
        json.dump(results, f, indent=2)
 
    report_file = OUTPUT_DIR / "cross_check_report.json"
    with open(report_file, "w") as f:
        json.dump(cross_check_report, f, indent=2)
 
    print(f"\n  Master JSON    : {master_file}")
    print(f"  Cross-Check    : {report_file}")
    print("=" * 65)
 
    return results, cross_check_report
 

In [18]:
if __name__ == "__main__":
    run_all()
 


   NBFC OCR PIPELINE — Processing all documents

───────────────────────────────────────────────────────
  Processing: 01_Company_PAN_Card.png
  Doc Type  : PAN_CARD
───────────────────────────────────────────────────────
  🖼️  Image loaded: (1012, 638)
  📝 Page 1: 278 chars extracted
  🤖 Sending to Qwen3...
  ✅ JSON extracted: 5 fields
  💾 Saved: ocr_output\PAN_CARD_29BE211D.json

───────────────────────────────────────────────────────
  Processing: 02_GST_Certificate.pdf
  Doc Type  : GST_CERTIFICATE
───────────────────────────────────────────────────────
  📄 PDF loaded: 1 page(s)
  📝 Page 1: 1184 chars extracted
  🤖 Sending to Qwen3...
  ✅ JSON extracted: 11 fields
  💾 Saved: ocr_output\GST_CERTIFICATE_78C331B7.json

───────────────────────────────────────────────────────
  Processing: 03_LEI_Certificate.pdf
  Doc Type  : LEI_CERTIFICATE
───────────────────────────────────────────────────────
  📄 PDF loaded: 1 page(s)
  📝 Page 1: 1073 chars extracted
  🤖 Sending to Qwen3...
  ✅ JSO

---
# 🔐 API VERIFICATION LAYER
### Stage 2 — Verify extracted fields against real + sandbox APIs

**APIs used:**
| Field | Source | Type |
|---|---|---|
| LEI Code | GLEIF api.gleif.org | ✅ Real, free forever |
| Pincode | api.postalpincode.in | ✅ Real, free forever |
| PAN format | Local regex | ✅ No API needed |
| GSTIN format | Local regex | ✅ No API needed |
| CIN format | Local regex | ✅ No API needed |
| Company name match | difflib fuzzy | ✅ No API needed |
| PAN active/inactive | Sandbox mock | ✅ Simulates NSDL |
| GST active/inactive | Sandbox mock | ✅ Simulates GST Portal |
| CIN/MCA status | Sandbox mock | ✅ Simulates MCA21 |

In [ ]:
# ── Load OCR results from previous pipeline run ───────────
import json, re, httpx, difflib
from pathlib import Path
from datetime import datetime, timedelta

OCR_OUTPUT = Path("ocr_output")

def load_ocr_results() -> list:
    master = OCR_OUTPUT / "all_documents.json"
    if not master.exists():
        raise FileNotFoundError("Run the OCR pipeline first — all_documents.json not found")
    with open(master) as f:
        docs = json.load(f)
    print(f"✅ Loaded {len(docs)} document(s) from OCR output")
    for d in docs:
        print(f"   {d['doc_type']:<40} status={d['status']}")
    return docs

OCR_DOCS = load_ocr_results()

# Helper — get extracted field from a specific doc type
def get_field(doc_type: str, field: str):
    for d in OCR_DOCS:
        if d['doc_type'] == doc_type:
            return d.get('fields', {}).get(field) or d.get('cross_check', {}).get(field)
    return None

# Pull key fields extracted by OCR
EXTRACTED = {
    'pan':     get_field('PAN_CARD', 'pan_number')     or get_field('GST_CERTIFICATE', 'pan_number'),
    'gstin':   get_field('GST_CERTIFICATE', 'gstin'),
    'lei':     get_field('LEI_CERTIFICATE', 'lei_code'),
    'cin':     get_field('INCORPORATION_CERTIFICATE', 'cin'),
    'pincode': get_field('REGISTERED_ADDRESS', 'pincode') or get_field('GST_CERTIFICATE', 'pincode'),
    'company': get_field('GST_CERTIFICATE', 'legal_name'),
    'bill_date': get_field('ELECTRICITY_BILL', 'bill_date'),
}

print("\n📋 Key fields for verification:")
for k, v in EXTRACTED.items():
    print(f"   {k:<12} : {v}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# SANDBOX — Simulates NSDL, GST Portal, MCA21
# In production: swap these calls with real API endpoints
# ═══════════════════════════════════════════════════════════

SANDBOX = {
    'pan': {
        'AAKCM1234C': {'name': 'MANIKANDAN CONSTRUCTIONS PVT LTD', 'status': 'ACTIVE',   'type': 'Company'},
        'BNZPM2501F': {'name': 'D MANIKANDAN',                     'status': 'ACTIVE',   'type': 'Individual'},
        'ZZZZZ9999Z': {'name': 'SHELL CORP LTD',                   'status': 'INACTIVE', 'type': 'Company'},
    },
    'gst': {
        '33AAKCM1234C1ZP': {'legal_name': 'MANIKANDAN CONSTRUCTIONS PVT LTD', 'status': 'Active',    'state': 'Tamil Nadu', 'state_code': '33', 'reg_date': '15/04/2015'},
        '27ZZZZZ9999Z1Z5': {'legal_name': 'SHELL CORP LTD',                   'status': 'Cancelled', 'state': 'Maharashtra'},
    },
    'cin': {
        'U45200TN2015PTC123456': {'company_name': 'MANIKANDAN CONSTRUCTIONS PVT LTD', 'status': 'Active', 'roc': 'ROC Chennai', 'incorp_date': '15/04/2015'},
        'U45200MH2010PTC999999': {'company_name': 'SHELL CORP LTD',                   'status': 'Strike Off', 'roc': 'ROC Mumbai'},
    }
}

def sandbox_pan(pan: str) -> dict:
    data = SANDBOX['pan'].get(pan)
    if not data:
        return {'verified': False, 'status': 'NOT_FOUND', 'source': 'SANDBOX', 'pan': pan}
    return {'verified': data['status'] == 'ACTIVE', 'status': data['status'],
            'name': data['name'], 'type': data['type'], 'source': 'SANDBOX', 'pan': pan}

def sandbox_gst(gstin: str) -> dict:
    data = SANDBOX['gst'].get(gstin)
    if not data:
        return {'verified': False, 'status': 'NOT_FOUND', 'source': 'SANDBOX', 'gstin': gstin}
    return {'verified': data['status'] == 'Active', 'status': data['status'],
            'legal_name': data['legal_name'], 'state': data['state'],
            'state_code': data['state_code'], 'source': 'SANDBOX', 'gstin': gstin}

def sandbox_cin(cin: str) -> dict:
    data = SANDBOX['cin'].get(cin)
    if not data:
        return {'verified': False, 'status': 'NOT_FOUND', 'source': 'SANDBOX', 'cin': cin}
    return {'verified': data['status'] == 'Active', 'status': data['status'],
            'company_name': data['company_name'], 'roc': data['roc'],
            'incorp_date': data['incorp_date'], 'source': 'SANDBOX', 'cin': cin}

print('✅ Sandbox DB ready (PAN, GST, CIN)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# FORMAT VALIDATORS — local, no API needed
# ═══════════════════════════════════════════════════════════

def validate_pan_format(pan: str) -> dict:
    """PAN: 5 letters, 4 digits, 1 letter. 4th char = entity type."""
    if not pan:
        return {'valid': False, 'error': 'PAN is None/empty'}
    pan = pan.strip().upper()
    pattern = re.compile(r'^[A-Z]{5}[0-9]{4}[A-Z]$')
    if not pattern.match(pan):
        return {'valid': False, 'pan': pan, 'error': 'Invalid PAN format'}
    entity_map = {'P': 'Individual', 'C': 'Company', 'H': 'HUF', 'F': 'Firm',
                  'A': 'AOP', 'T': 'Trust', 'B': 'BOI', 'G': 'Govt', 'J': 'AJP'}
    entity = entity_map.get(pan[3], 'Unknown')
    return {'valid': True, 'pan': pan, 'entity_type_code': pan[3],
            'entity_type': entity, 'is_company': pan[3] == 'C'}

def validate_gstin_format(gstin: str) -> dict:
    """GSTIN: 15 chars. State code (2) + PAN (10) + entity (1) + Z + checksum."""
    if not gstin:
        return {'valid': False, 'error': 'GSTIN is None/empty'}
    gstin = gstin.strip().upper()
    pattern = re.compile(r'^[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z][1-9A-Z]Z[0-9A-Z]$')
    if not pattern.match(gstin):
        return {'valid': False, 'gstin': gstin, 'error': 'Invalid GSTIN format'}
    state_codes = {
        '01':'Jammu & Kashmir','02':'Himachal Pradesh','03':'Punjab','04':'Chandigarh',
        '05':'Uttarakhand','06':'Haryana','07':'Delhi','08':'Rajasthan','09':'Uttar Pradesh',
        '10':'Bihar','11':'Sikkim','12':'Arunachal Pradesh','13':'Nagaland','14':'Manipur',
        '15':'Mizoram','16':'Tripura','17':'Meghalaya','18':'Assam','19':'West Bengal',
        '20':'Jharkhand','21':'Odisha','22':'Chhattisgarh','23':'Madhya Pradesh',
        '24':'Gujarat','26':'Daman & Diu','27':'Maharashtra','29':'Karnataka',
        '30':'Goa','31':'Lakshadweep','32':'Kerala','33':'Tamil Nadu','34':'Puducherry',
        '35':'Andaman & Nicobar','36':'Telangana','37':'Andhra Pradesh'
    }
    sc   = gstin[:2]
    pan  = gstin[2:12]
    state = state_codes.get(sc, 'Unknown')
    return {'valid': True, 'gstin': gstin, 'state_code': sc,
            'state': state, 'embedded_pan': pan, 'format_ok': True}

def validate_cin_format(cin: str) -> dict:
    """CIN: U/L + 5 digits + 2 state letters + 4 year + 3 type letters + 6 digits."""
    if not cin:
        return {'valid': False, 'error': 'CIN is None/empty'}
    cin = cin.strip().upper()
    pattern = re.compile(r'^[UL][0-9]{5}[A-Z]{2}[0-9]{4}[A-Z]{3}[0-9]{6}$')
    if not pattern.match(cin):
        return {'valid': False, 'cin': cin, 'error': 'Invalid CIN format'}
    return {'valid': True, 'cin': cin,
            'listing_status': 'Listed' if cin[0] == 'L' else 'Unlisted',
            'industry_code': cin[1:6],
            'state_code': cin[6:8],
            'year_of_incorp': cin[8:12],
            'company_type': cin[12:15]}

def validate_lei_format(lei: str) -> dict:
    """LEI: 20 alphanumeric characters."""
    if not lei:
        return {'valid': False, 'error': 'LEI is None/empty'}
    lei = lei.strip().upper()
    if not re.match(r'^[A-Z0-9]{20}$', lei):
        return {'valid': False, 'lei': lei, 'error': 'Invalid LEI format (must be 20 alphanumeric chars)'}
    return {'valid': True, 'lei': lei, 'lou_prefix': lei[:4]}

# Test all format validators
print('=== FORMAT VALIDATION TEST ===')
print('PAN :', validate_pan_format(EXTRACTED.get('pan', '')))
print('GST :', validate_gstin_format(EXTRACTED.get('gstin', '')))
print('CIN :', validate_cin_format(EXTRACTED.get('cin', '')))
print('LEI :', validate_lei_format(EXTRACTED.get('lei', '')))

In [ ]:
# ═══════════════════════════════════════════════════════════
# PAN ↔ GSTIN CROSS-EMBED CHECK
# GSTIN chars 3–12 must equal PAN. No API needed.
# ═══════════════════════════════════════════════════════════

def check_pan_gstin_embed(pan: str, gstin: str) -> dict:
    """PAN embedded in GSTIN at positions 3–12 must match."""
    if not pan or not gstin:
        return {'consistent': False, 'error': 'PAN or GSTIN missing'}
    pan   = pan.strip().upper()
    gstin = gstin.strip().upper()
    embedded = gstin[2:12]
    match = embedded == pan
    return {
        'consistent':    match,
        'pan':           pan,
        'gstin':         gstin,
        'embedded_pan':  embedded,
        'message':       'PAN correctly embedded in GSTIN' if match else f'MISMATCH: PAN={pan} but GSTIN embeds={embedded}'
    }

result = check_pan_gstin_embed(EXTRACTED.get('pan'), EXTRACTED.get('gstin'))
print('PAN ↔ GSTIN Embed Check:')
print(f"  Consistent : {result['consistent']}")
print(f"  Message    : {result['message']}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# GLEIF — Real Free API. No key, no registration, forever free.
# https://api.gleif.org/api/v1/lei-records/{LEI}
# ═══════════════════════════════════════════════════════════

def verify_lei_gleif(lei: str, expected_name: str = None) -> dict:
    """Verify LEI against GLEIF. Completely free, no auth."""
    if not lei:
        return {'verified': False, 'error': 'No LEI provided', 'source': 'GLEIF'}
    lei = lei.strip().upper()
    url = f'https://api.gleif.org/api/v1/lei-records/{lei}'
    try:
        resp = httpx.get(url, timeout=10.0)
        if resp.status_code == 404:
            return {'verified': False, 'status': 'NOT_FOUND', 'lei': lei, 'source': 'GLEIF'}
        if resp.status_code != 200:
            return {'verified': False, 'error': f'HTTP {resp.status_code}', 'source': 'GLEIF'}
        data        = resp.json()['data']['attributes']
        reg         = data['registration']
        entity      = data['entity']
        legal_name  = entity['legalName']['name']
        status      = reg['status']
        renewal     = reg.get('nextRenewalDate', 'N/A')
        # Name match check
        name_match = True
        if expected_name:
            similarity = difflib.SequenceMatcher(None, legal_name.lower(), expected_name.lower()).ratio()
            name_match = similarity > 0.80
        return {
            'verified':         status == 'ISSUED',
            'lei':              lei,
            'legal_name':       legal_name,
            'status':           status,
            'next_renewal':     renewal,
            'corroboration':    reg.get('corroborationLevel', 'N/A'),
            'name_match':       name_match,
            'source':           'GLEIF'
        }
    except httpx.ConnectError:
        # Fallback if no internet — use format check only
        fmt = validate_lei_format(lei)
        return {'verified': fmt['valid'], 'lei': lei,
                'note': 'No internet — format check only', 'source': 'GLEIF_OFFLINE'}
    except Exception as e:
        return {'verified': False, 'error': str(e), 'source': 'GLEIF'}

lei_result = verify_lei_gleif(EXTRACTED.get('lei'), EXTRACTED.get('company'))
print('GLEIF LEI Verification:')
for k, v in lei_result.items():
    print(f'  {k:<20} : {v}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# INDIA PINCODE API — Free, no auth
# https://api.postalpincode.in/pincode/{pincode}
# ═══════════════════════════════════════════════════════════

def verify_pincode(pincode: str, expected_state: str = None, expected_city: str = None) -> dict:
    """Verify Indian pincode. Free, no API key."""
    if not pincode:
        return {'verified': False, 'error': 'No pincode provided'}
    pincode = str(pincode).strip()
    if not re.match(r'^[1-9][0-9]{5}$', pincode):
        return {'verified': False, 'pincode': pincode, 'error': 'Invalid pincode format'}
    try:
        resp = httpx.get(f'https://api.postalpincode.in/pincode/{pincode}', timeout=8.0)
        data = resp.json()
        if not data or data[0]['Status'] != 'Success':
            return {'verified': False, 'pincode': pincode, 'error': 'Pincode not found'}
        post_offices = data[0]['PostOffice']
        if not post_offices:
            return {'verified': False, 'pincode': pincode, 'error': 'No post offices found'}
        po    = post_offices[0]
        state = po.get('State', '')
        dist  = po.get('District', '')
        # State match
        state_match = True
        if expected_state:
            state_match = expected_state.lower() in state.lower() or state.lower() in expected_state.lower()
        return {
            'verified':     True,
            'pincode':      pincode,
            'state':        state,
            'district':     dist,
            'region':       po.get('Region', ''),
            'post_offices': len(post_offices),
            'state_match':  state_match,
            'source':       'INDIA_POSTAL_API'
        }
    except httpx.ConnectError:
        return {'verified': re.match(r'^[1-9][0-9]{5}$', pincode) is not None,
                'pincode': pincode, 'note': 'No internet — format check only'}
    except Exception as e:
        return {'verified': False, 'error': str(e)}

pin_result = verify_pincode(EXTRACTED.get('pincode'), expected_state='Tamil Nadu')
print('Pincode Verification:')
for k, v in pin_result.items():
    print(f'  {k:<20} : {v}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# SANDBOX API CALLS — PAN, GST, CIN
# Simulates NSDL, GST Portal, MCA21
# Production: swap function body with real httpx.get/post calls
# ═══════════════════════════════════════════════════════════

pan_result = sandbox_pan(EXTRACTED.get('pan', ''))
gst_result = sandbox_gst(EXTRACTED.get('gstin', ''))
cin_result = sandbox_cin(EXTRACTED.get('cin', ''))

print('PAN  (Sandbox/NSDL)  :', pan_result)
print('GST  (Sandbox/Portal):', gst_result)
print('CIN  (Sandbox/MCA21) :', cin_result)

In [ ]:
# ═══════════════════════════════════════════════════════════
# COMPANY NAME FUZZY MATCH
# Checks if names across all API responses are consistent.
# Handles 'PVT LTD' vs 'PRIVATE LIMITED' type variations.
# ═══════════════════════════════════════════════════════════

def normalize_company_name(name: str) -> str:
    if not name: return ''
    n = name.upper().strip()
    replacements = [
        ('PRIVATE LIMITED', 'PVT LTD'),
        ('PVT. LTD.', 'PVT LTD'),
        ('PVT.LTD.', 'PVT LTD'),
        ('LIMITED', 'LTD'),
        ('LTD.', 'LTD'),
    ]
    for old, new in replacements:
        n = n.replace(old, new)
    return ' '.join(n.split())

def fuzzy_name_match(names: dict, threshold: float = 0.82) -> dict:
    """
    names = {'source_label': 'Company Name'}
    Returns consistency check across all sources.
    """
    normalized = {src: normalize_company_name(n) for src, n in names.items() if n}
    results    = {}
    base_src, base_name = list(normalized.items())[0]
    all_match = True
    for src, name in normalized.items():
        score = difflib.SequenceMatcher(None, base_name, name).ratio()
        match = score >= threshold
        if not match: all_match = False
        results[src] = {'name': name, 'score': round(score, 3), 'match': match}
    return {'consistent': all_match, 'base': base_name, 'details': results, 'threshold': threshold}

# Collect names from all sources
names_to_check = {
    'OCR_PAN':        get_field('PAN_CARD', 'entity_name'),
    'OCR_GST':        get_field('GST_CERTIFICATE', 'legal_name'),
    'OCR_LEI':        get_field('LEI_CERTIFICATE', 'legal_name'),
    'OCR_INCORP':     get_field('INCORPORATION_CERTIFICATE', 'company_name'),
    'SANDBOX_PAN':    pan_result.get('name'),
    'SANDBOX_GST':    gst_result.get('legal_name'),
    'SANDBOX_CIN':    cin_result.get('company_name'),
    'GLEIF':          lei_result.get('legal_name'),
}
# Filter None values
names_to_check = {k: v for k, v in names_to_check.items() if v}

name_result = fuzzy_name_match(names_to_check)
print(f"Company Name Consistency (threshold={name_result['threshold']})")
print(f"  Overall Consistent : {name_result['consistent']}")
print(f"  Base Name          : {name_result['base']}")
print()
for src, d in name_result['details'].items():
    mark = '✅' if d['match'] else '❌'
    print(f"  {mark} {src:<20} score={d['score']:.2f}  name={d['name']}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# BILL DATE VALIDATION
# Electricity/Telephone bills must be within 90 days.
# NBFC compliance standard.
# ═══════════════════════════════════════════════════════════

def validate_bill_date(bill_date_str: str, max_days: int = 90) -> dict:
    """Check if bill date is within allowed window (default 90 days)."""
    if not bill_date_str:
        return {'valid': False, 'error': 'No bill date found'}
    formats = ['%d/%m/%Y', '%d-%m-%Y', '%Y-%m-%d', '%d/%m/%y']
    parsed  = None
    for fmt in formats:
        try:
            parsed = datetime.strptime(bill_date_str.strip(), fmt)
            break
        except ValueError:
            continue
    if not parsed:
        return {'valid': False, 'error': f'Cannot parse date: {bill_date_str}'}
    today    = datetime.now()
    age_days = (today - parsed).days
    valid    = 0 <= age_days <= max_days
    return {
        'valid':      valid,
        'bill_date':  bill_date_str,
        'age_days':   age_days,
        'max_days':   max_days,
        'message':    f'Bill is {age_days} days old — {"WITHIN" if valid else "EXCEEDS"} {max_days}-day limit'
    }

elec_date = get_field('ELECTRICITY_BILL', 'bill_date')
tel_date  = get_field('TELEPHONE_BILL',  'bill_date')
print('Electricity Bill Date:', validate_bill_date(elec_date))
print('Telephone  Bill Date :', validate_bill_date(tel_date))

In [ ]:
# ═══════════════════════════════════════════════════════════
# MASTER VERIFICATION RUNNER
# Runs all checks and produces a final verification report.
# ═══════════════════════════════════════════════════════════

def run_api_verification() -> dict:
    print('\n' + '='*60)
    print('   API VERIFICATION REPORT')
    print('='*60)
    report = {'timestamp': datetime.now().isoformat(), 'checks': {}, 'passed': 0, 'failed': 0, 'warnings': 0}

    def add(name, result, critical=True):
        passed   = result.get('verified') or result.get('valid') or result.get('consistent') or False
        report['checks'][name] = {**result, 'critical': critical}
        if passed:
            report['passed']  += 1
            print(f'  ✅  {name}')
        else:
            if critical:
                report['failed'] += 1
                print(f'  ❌  {name}')
            else:
                report['warnings'] += 1
                print(f'  ⚠️   {name} (non-critical)')
        # Print key detail
        for key in ['error','message','status','state','note']:
            if result.get(key):
                print(f'       → {result[key]}')
                break

    print('\n  ── FORMAT CHECKS (local) ──────────────────────────')
    add('PAN Format',         validate_pan_format(EXTRACTED.get('pan',   '')))
    add('GSTIN Format',       validate_gstin_format(EXTRACTED.get('gstin','')))
    add('CIN Format',         validate_cin_format(EXTRACTED.get('cin',  '')))
    add('LEI Format',         validate_lei_format(EXTRACTED.get('lei',  '')))
    add('PAN ↔ GSTIN Embed',  check_pan_gstin_embed(EXTRACTED.get('pan'), EXTRACTED.get('gstin')))

    print('\n  ── REAL APIs ──────────────────────────────────────')
    add('LEI (GLEIF)',         verify_lei_gleif(EXTRACTED.get('lei'), EXTRACTED.get('company')))
    add('Pincode (Postal API)',verify_pincode(EXTRACTED.get('pincode'), expected_state='Tamil Nadu'))

    print('\n  ── SANDBOX (simulates NSDL, GST Portal, MCA21) ───')
    add('PAN Active (Sandbox)',sandbox_pan(EXTRACTED.get('pan', '')))
    add('GST Active (Sandbox)',sandbox_gst(EXTRACTED.get('gstin', '')))
    add('CIN Active (Sandbox)',sandbox_cin(EXTRACTED.get('cin', '')))

    print('\n  ── NAME CONSISTENCY ───────────────────────────────')
    names = {k: v for k, v in {
        'OCR_GST': get_field('GST_CERTIFICATE', 'legal_name'),
        'SANDBOX_PAN': sandbox_pan(EXTRACTED.get('pan','')).get('name'),
        'SANDBOX_GST': sandbox_gst(EXTRACTED.get('gstin','')).get('legal_name'),
        'GLEIF': verify_lei_gleif(EXTRACTED.get('lei'), None).get('legal_name'),
    }.items() if v}
    add('Company Name Consistency', fuzzy_name_match(names), critical=False)

    print('\n  ── BILL DATE CHECKS ───────────────────────────────')
    add('Electricity Bill Date', validate_bill_date(get_field('ELECTRICITY_BILL','bill_date')), critical=False)
    add('Telephone Bill Date',   validate_bill_date(get_field('TELEPHONE_BILL','bill_date')),   critical=False)

    # Final verdict
    print('\n' + '='*60)
    print(f"  ✅ Passed   : {report['passed']}")
    print(f"  ❌ Failed   : {report['failed']}")
    print(f"  ⚠️  Warnings : {report['warnings']}")
    if report['failed'] == 0:
        print('\n  VERDICT: ✅ ALL CRITICAL CHECKS PASSED')
        report['verdict'] = 'PASS'
    else:
        print(f"\n  VERDICT: ❌ {report['failed']} CRITICAL CHECK(S) FAILED — Do not proceed")
        report['verdict'] = 'FAIL'
    print('='*60)

    # Save report
    out = Path('ocr_output') / 'api_verification_report.json'
    with open(out, 'w') as f:
        json.dump(report, f, indent=2)
    print(f'\n  💾 Report saved: {out}')
    return report

VERIFICATION_REPORT = run_api_verification()